# Synonym RxNorm Acquisition

## Check Python Version

In [1]:
!python --version

Python 3.11.17


## Setup Config

In [2]:
PROJECT_DIR = "../../"

import pathlib
OUTPUT_DIR = pathlib.Path(PROJECT_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_RX_INGREDIENTS = "rxnorm/rx_norm_ingredients.csv"
RX_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_INGREDIENTS)
print(f"RX Ingredients Dataset Location: {RX_INGREDIENTS_PATH}")

FILE_RX_SYNONYM_INGREDIENTS = "rxnorm/rx_norm_ingredient_synonyms.csv"
RESOLVED_SYNONYM_PATH = OUTPUT_DIR.joinpath(FILE_RX_SYNONYM_INGREDIENTS)
print(f"Resolved Rx Synonym Ingredients Location: {RESOLVED_SYNONYM_PATH}")

Dataset Directory: ../..
RX Ingredients Dataset Location: ../../rxnorm/rx_norm_ingredients.csv
Resolved Rx Synonym Ingredients Location: ../../rxnorm/rx_norm_ingredient_synonyms.csv


## Read Unique Drug from DDI Dataset

In [3]:
import requests
import pandas as pd
from tqdm import tqdm

df = pd.read_csv(RX_INGREDIENTS_PATH)
print(f"Total Ingredients need to be resolved: {df.shape[0]}")

Total Ingredients need to be resolved: 14718


## Synonym Resolver using RxNorm

In [4]:
import time
import requests

class RxNormSynonymResolver:
    ALLOWED_TYPES = {"SY", "FSY"}
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def _get_rxcui(self, drug_name: str):
        try:
            url = f"{self.BASE_URL}/rxcui.json"
    
            response = requests.get(
                url,
                params={"name": drug_name},
                timeout=10
            )
    
            response.raise_for_status()
            data = response.json()
    
            rxnorm_ids = (
                data.get("idGroup", {})
                .get("rxnormId", [])
            )
    
            if not rxnorm_ids:
                return None
    
            return rxnorm_ids[0]
    
        except Exception as e:
            print(f"[RXCUI ERROR] {drug_name}: {e}")
            return None


    def _get_aliases(self, name: str, rxcui: str):
        try:
            url = f"{self.BASE_URL}/rxcui/{rxcui}/proprietary.json"
    
            response = requests.get(url, timeout=10)
            response.raise_for_status()
    
            data = response.json()
    
            proprietary_info = (
                data.get("proprietaryGroup", {})
                .get("proprietaryInfo", [])
            )
    
            aliases = {}

            for item in proprietary_info:
                if item.get("type") not in self.ALLOWED_TYPES:
                    continue
    
                proprietary_name = item.get("name")
                source = item.get("source")
    
                if not proprietary_name:
                    continue
    
                if name.lower() == proprietary_name.lower():
                    continue
    
                key = proprietary_name.lower()

                if key in aliases:
                    a = aliases[key]
                    a["source"].append(source)

                    aliases[key] = a
                else:
                    aliases[key] = {
                        "name": key,
                        "source": [source]
                    }
    
            return {
                "rx_cui": rxcui,
                "name": name,
                "aliases": list(aliases.values())
            }
    
        except Exception as e:
            print(f"[ALIAS ERROR] {rxcui}: {e}")
    
            return None

    def get_synonyms_by_rx_cui(self, rxcui, drug):
        alias_data = self._get_aliases(drug, rxcui)
        if not alias_data:
            return {
                "status": "unknown_aliases",
                "data": {
                    "rxcui": rxcui,
                    "name": drug
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "rxcui": rxcui,
                "name": drug,
                "aliases": alias_data["aliases"]
            }
        }
        
    def get_synonyms(self, name):
        drug = str(name).strip().lower()
    
        rxcui = self._get_rxcui(drug)
        if not rxcui:
            return {
                "status": "unknown_rxcui",
                "data": {
                    "name": drug
                }
            }
    
        alias_data = self._get_aliases(drug, rxcui)
        if not alias_data:
            return {
                "status": "unknown_aliases",
                "data": {
                    "name": drug
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "name": drug, 
                "rxcui": rxcui,
                "aliases": alias_data["aliases"]
            }
        }

## Resolve Synonyms for All IN type in RxNorm

In [5]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


results = []
unknown_aliases = []

# Maksimal 5 request diproses secara paralel
resolver = RxNormSynonymResolver()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            resolver.get_synonyms_by_rx_cui,
            row["rx_cui"],
            row["name"]
        ): row
        for idx, row in df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):
        try:
            result = future.result()

            if result["status"] == "ok":
                results.append(result["data"])

            elif result["status"] == "unknown_aliases":
                unknown_aliases.append(result["data"])

        except Exception as e:
            print(f"[ERROR] {futures[future]}: {e}")

100%|████████████████████████████████████████████████| 14718/14718 [1:10:55<00:00,  3.46it/s]


In [6]:
# Sort lagi karena parallel processing
# membuat urutan hasil tidak terjamin
results.sort(key=lambda x: x["rxcui"])
unknown_aliases.sort(key=lambda x: x["rxcui"])

In [7]:
# Export result
df_result = pd.DataFrame(results)
df_result.to_csv(RESOLVED_SYNONYM_PATH, index=False)

print(f"Success         : {len(results)}")
print(f"Unknown aliases : {len(unknown_aliases)}")

Success         : 14718
Unknown aliases : 0
